# box-array-to-tensor-with-recipe — ex2: wrap_forward_fn: compute requires_grad gate, then box the raw output

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `box-array-to-tensor-with-recipe`. Running the final beacon cell reports progress against the `Backprop: Box array as Tensor + recipe` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    """Minimal Tensor wrapper for the ARENA-style manual-autograd drills.
    Wraps a raw `torch.Tensor` in `.array`. Carries optional `.recipe`,
    `.requires_grad`, and `.grad` (the accumulated gradient at leaves)."""
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
        self.grad = None
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Backprop: Box array as Tensor + recipe` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`box-array-to-tensor-with-recipe`** (exercise 2). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "box-array-to-tensor-with-recipe"
DD_SUBTOPIC = "Backprop: Box array as Tensor + recipe"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Box step — computing the requires_grad gate — quick refresher

`box_with_recipe` from ex1 receives `requires_grad` as a precomputed bool. Who computes it? The OUTER `wrap_forward_fn`, by combining two signals:

1. **Any-input gate.** `any(isinstance(a, MiniTensor) and a.requires_grad for a in args)`. If a single input is grad-tracked, the output must be.
2. **Global toggle.** `grad_tracking_enabled` (a module-level bool). Inside an inference / no_grad block this is False — and overrides the any-input signal so nothing gets a Recipe.

The combined rule: `requires_grad = grad_tracking_enabled AND any(rg_input)`.

Then ex1's `box_with_recipe` does the actual boxing using this bool.

### Exercise 2 — wrap_forward_fn: compute requires_grad gate, then box the raw output

> ```yaml
> Difficulty: 🔴🔴🔴🔴⚪
> Bloom level: Apply
> LO: Apply the requires_grad gate at the wrapper boundary: any-input rg-bool AND the global grad-tracking toggle, then box-with-recipe.
> Keywords: wrap-forward, requires-grad-gate, global-toggle, box, recipe
> ```

**KCs targeted:** `box-array-to-tensor-with-recipe`, `recipe-dataclass`

Implement `wrap_forward_fn(fwd_fn)` — the FULL wrapper factory. Given a raw forward fn (e.g. `torch.log`), return a new function `wrapped(*args, **kwargs)` that:

1. **Unbox** positional args: `raw_args = tuple(a.array if isinstance(a, MiniTensor) else a for a in args)`.
2. **Compute the gate**: `requires_grad = grad_tracking_enabled and any(isinstance(a, MiniTensor) and a.requires_grad for a in args)`.
3. **Run the raw forward**: `out_raw = fwd_fn(*raw_args, **kwargs)`.
4. **Build parents** (only if grad-tracked): `{idx: a for idx, a in enumerate(args) if isinstance(a, MiniTensor) and a.requires_grad}`.
5. **Box** with Recipe iff `requires_grad`: construct `MiniTensor(out_raw, requires_grad=requires_grad)`, attach `Recipe(fwd_fn, raw_args, kwargs, parents)` iff True.

**Why this is harder than ex1.** Ex1's `box_with_recipe` was the FINAL step — handed all the bookkeeping. Here you compose the WHOLE wrapper. The trap is the gate: two signals (`any-input` AND `global-toggle`) combine with AND. Forgetting the toggle means `no_grad()` blocks silently still build the graph; forgetting the any-input check means every constant + constant call gets a useless Recipe.

Signature: `wrap_forward_fn(fwd_fn) -> Callable`. The returned callable takes `*args, **kwargs` and returns a `MiniTensor`.

In [ ]:
def wrap_forward_fn(fwd_fn):
    def wrapped(*args, **kwargs):
        # 1. Unbox positional args (raw fn doesn't know MiniTensor)
        raw_args = tuple(
            a.array if isinstance(a, MiniTensor) else a for a in args
        )
        # 2. Gate: global toggle AND any-rg-input
        any_rg = any(
            isinstance(a, MiniTensor) and a.requires_grad for a in args
        )
        requires_grad = grad_tracking_enabled and any_rg
        # 3. Run raw forward
        out_raw = fwd_fn(*raw_args, **kwargs)
        # 4. Build parents only if grad-tracked (skip-graph optimization)
        parents = {}
        if requires_grad:
            parents = {
                idx: a for idx, a in enumerate(args)
                if isinstance(a, MiniTensor) and a.requires_grad
            }
        # 5. Box (+ Recipe iff grad-tracked)
        out = MiniTensor(out_raw, requires_grad=requires_grad)
        if requires_grad:
            out.recipe = Recipe(fwd_fn, raw_args, kwargs, parents)
        return out
    return wrapped


<details><summary>Solution</summary>

```python
def wrap_forward_fn(fwd_fn):
    def wrapped(*args, **kwargs):
        # 1. Unbox positional args (raw fn doesn't know MiniTensor)
        raw_args = tuple(
            a.array if isinstance(a, MiniTensor) else a for a in args
        )
        # 2. Gate: global toggle AND any-rg-input
        any_rg = any(
            isinstance(a, MiniTensor) and a.requires_grad for a in args
        )
        requires_grad = grad_tracking_enabled and any_rg
        # 3. Run raw forward
        out_raw = fwd_fn(*raw_args, **kwargs)
        # 4. Build parents only if grad-tracked (skip-graph optimization)
        parents = {}
        if requires_grad:
            parents = {
                idx: a for idx, a in enumerate(args)
                if isinstance(a, MiniTensor) and a.requires_grad
            }
        # 5. Box (+ Recipe iff grad-tracked)
        out = MiniTensor(out_raw, requires_grad=requires_grad)
        if requires_grad:
            out.recipe = Recipe(fwd_fn, raw_args, kwargs, parents)
        return out
    return wrapped
```

**The two-signal gate is the load-bearing facet.** Ex1 got `requires_grad` as a bool — here you have to compute it. Forget the global toggle and `no_grad()` is a no-op. Forget the any-input check and constant + constant calls get useless Recipes.

**Why build parents AFTER the gate.** When grad-tracking is off, building parents is wasted work (the Recipe is never attached, so the dict is dropped). The conditional keeps inference paths allocation-free.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex2',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()